# LOO summary table (DEG 200)

Builds a NeurIPS-ready summary table from `results/loo_summary_deg_200.csv`.

- Aggregates across seeds and held-out cell types (mean ± std).
- Bolds the best performer per metric (direction-aware).
- Renames `baseline` to `mean shift`.
- Annotates columns with ↑ (higher is better) or ↓ (lower is better).

In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
DATASET_NAME = "merfish"
N_DEG = 50
CSV_PATH = f"../results/loo_summary_{DATASET_NAME}_DEG_{N_DEG}.csv"
OUT_DIR = Path("../results")
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CSV_PATH, engine="python")
df.head()

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,direction_match_gt,mixing_index,edistance_global,edistance_local,edistance_pca_log,edistance_pca,rmse
0,C57BL6J-2.036,baseline,GABAergic neuron_Fiber_tracts,50,0.574783,0.676753,0.00,0.0,0.00,0.08,0.019022,42.316212,43.006039,27.583193,1517.932397,2.787898
1,C57BL6J-2.036,baseline,GABAergic neuron_Isocortex,50,0.656297,0.672887,0.20,0.9,0.18,0.60,0.293658,48.246544,52.527597,29.113398,1110.563318,3.173243
2,C57BL6J-2.036,baseline,astrocyte_Fiber_tracts,50,0.301952,0.354013,0.18,1.0,0.18,0.82,0.069700,38.690306,37.781634,17.030342,1414.752209,3.271581
3,C57BL6J-2.036,baseline,astrocyte_Isocortex,50,0.774857,0.715261,0.26,1.0,0.26,0.84,0.000000,64.490029,66.906740,37.627463,2108.755310,3.473014
4,C57BL6J-2.036,baseline,endothelial cell_Fiber_tracts,50,0.742315,0.662692,0.06,1.0,0.06,0.86,0.003236,43.548324,43.732465,21.867013,1863.143823,3.222337


In [4]:
# Rename holdout_celltype by replacing the last '_' in with '-'
if DATASET_NAME == "merfish":
    df["holdout_celltype"] = df["holdout_celltype"].str.replace("Fiber_tracts", "Fiber-tracts", regex=False)
else:
    df["holdout_celltype"] = df["holdout_celltype"].str.replace("T_cell", "T-cell", regex=False)

In [5]:
# Split holdout_celltype on '_' and place everything in [0] as holdout_celltype, and everything in [1:] as perturbation
df["perturbation"] = df["holdout_celltype"].apply(lambda x: "".join(x.split("_")[-1]) if len(x.split("_")) > 1 else "")
df["holdout_celltype"] = df["holdout_celltype"].apply(lambda x: "-".join(x.split("_")[:-1]))
df

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,direction_match_gt,mixing_index,edistance_global,edistance_local,edistance_pca_log,edistance_pca,rmse,perturbation
0,C57BL6J-2.036,baseline,GABAergic neuron,50,0.574783,0.676753,0.00,0.0,0.00,0.08,0.019022,42.316212,43.006039,27.583193,1517.932397,2.787898,Fiber-tracts
1,C57BL6J-2.036,baseline,GABAergic neuron,50,0.656297,0.672887,0.20,0.9,0.18,0.60,0.293658,48.246544,52.527597,29.113398,1110.563318,3.173243,Isocortex
2,C57BL6J-2.036,baseline,astrocyte,50,0.301952,0.354013,0.18,1.0,0.18,0.82,0.069700,38.690306,37.781634,17.030342,1414.752209,3.271581,Fiber-tracts
3,C57BL6J-2.036,baseline,astrocyte,50,0.774857,0.715261,0.26,1.0,0.26,0.84,0.000000,64.490029,66.906740,37.627463,2108.755310,3.473014,Isocortex
4,C57BL6J-2.036,baseline,endothelial cell,50,0.742315,0.662692,0.06,1.0,0.06,0.86,0.003236,43.548324,43.732465,21.867013,1863.143823,3.222337,Fiber-tracts
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
293,C57BL6J-2.041,spatialprop-pert,astrocyte,50,0.725618,0.743672,0.06,1.0,0.06,0.90,0.067684,44.806450,52.293761,17.152685,400.291437,3.455386,Fiber-tracts
294,C57BL6J-2.041,spatialprop-pert,oligodendrocyte,50,0.399952,0.639537,0.12,1.0,0.12,0.96,0.156538,57.012378,66.244428,18.468198,477.076672,3.139007,Isocortex
295,C57BL6J-2.041,spatialprop-pert,oligodendrocyte,50,0.306795,0.660256,0.06,1.0,0.06,0.82,0.084819,52.042802,59.732181,18.682952,520.554156,3.831880,Fiber-tracts
296,C57BL6J-2.041,spatialprop-pert,endothelial cell,50,0.801200,0.870476,0.32,1.0,0.32,0.98,0.011535,56.382117,62.833591,19.648810,469.431580,3.420812,Isocortex


In [6]:
# Only keep 'Isocortex' perturbation
#df = df[df["perturbation"] == "Isocortex"]

In [7]:
# Metrics we want in the table and whether higher (+1) or lower (-1) is better.
METRICS = {
    "spearman": +1,
    "pearson": +1,
    #"precision": +1,
    "direction_match_k": +1,
    #"direction_match_gt": +1,
    #"edistance_local": -1,
    #"edistance_global": -1,
    "edistance_pca_log": -1,
    "rmse": -1,
    #"avg_rank": -1,
}

PRETTY_METRIC = {
    "spearman": r"Spearman $\uparrow$",
    "pearson": r"Pearson $\uparrow$",
    #"precision": r"Precision $\uparrow$",
    "direction_match_k": r"Signed Precision $\uparrow$",
    #"direction_match_gt": r"Direction Match$ (GT) \uparrow$",
    #"edistance_local": r"E-dist (local) $\downarrow$",
    #"edistance_global": r"E-dist (global) $\downarrow$",
    "edistance_pca_log": r"E-dist $\downarrow$",
    "rmse": r"RMSE $\downarrow$",
    #"avg_rank": r"Avg. Rank $\downarrow$",
}

# Model display order + renaming (baseline -> mean shift).
MODEL_RENAME = {
    "baseline": "mean shift",
    "cpa": "CPA",
    "scgen": "scGen",
    "spatialprop": "SpatialProp",
    "mintflow": "MintFlow",
    "cellina-ablated": "cellina (ablated)",
    "cellina-graph": "cellina-GAT",
    "cellina": "cellina",
    "cellina-pert": r"cellina$_{node-pert=200}$",
    "spatialprop-pert": r"SpatialProp$_{node-pert=200}$"
}
MODEL_ORDER = [
    "mean shift",
    "CPA",
    "scGen",
    "SpatialProp",
    "MintFlow",
    "cellina (ablated)",
    "cellina-GAT",
    "cellina",
    r"cellina$_{node-pert=200}$",
    r"SpatialProp$_{node-pert=200}$",
]

df["model_name"] = df["model_name"].map(lambda x: MODEL_RENAME.get(x, x))
df["model_name"].unique()

array(['mean shift', 'cellina (ablated)', 'cellina', 'cellina-GAT', 'CPA',
       'scGen', 'MintFlow', 'SpatialProp', 'cellina$_{node-pert=200}$',
       'SpatialProp$_{node-pert=200}$'], dtype=object)

In [8]:
# Two-step aggregation:
#   1) within each slide, average over held-out cell types
#   2) across slides, compute mean ± std
agg = (
    df.groupby(["perturbation", "model_name"])[list(METRICS)]
    .agg(["mean", "std"])
)

# Ensure a consistent model order within each perturbation. Build a
# MultiIndex with (perturbation, model) in the desired display order and
# reindex the aggregated table to that index.
perturbations = list(df["perturbation"].dropna().unique())
# keep stable ordering; treat empty string as its own group if present
if any(p == "" for p in perturbations):
    perturbations = sorted(perturbations, key=lambda x: (x != "", x))
else:
    perturbations = sorted(perturbations)

desired_index = pd.MultiIndex.from_product(
    [perturbations, MODEL_ORDER], names=["perturbation", "model_name"]
)
agg = agg.reindex(desired_index)

In [9]:
agg

spearman             pearson  \
                                                mean       std      mean   
perturbation model_name                                                    
Fiber-tracts mean shift                     0.351411  0.271832  0.360952   
             CPA                            0.584916  0.206990  0.798208   
             scGen                          0.496118  0.229757  0.715875   
             SpatialProp                    0.471744  0.204791  0.639798   
             MintFlow                       0.544253  0.228634  0.780124   
             cellina (ablated)              0.584583  0.215306  0.797372   
             cellina-GAT                    0.644486  0.153088  0.805590   
             cellina                        0.659409  0.213460  0.826450   
             cellina$_{node-pert=200}$      0.585005  0.229353  0.806803   
             SpatialProp$_{node-pert=200}$  0.496695  0.185900  0.661629   
Isocortex    mean shift                     0.594199  0.188672  0.575376   
             CPA                            0.781948  0.170337  0.837973   
             scGen                          0.719862  0.182515  0.820247   
             SpatialProp                    0.704227  0.123463  0.745628   
             MintFlow                       0.771640  0.184053  0.836621   
             cellina (ablated)              0.789042  0.162555  0.841151   
             cellina-GAT                    0.852629  0.110558  0.888902   
             cellina                        0.808256  0.177170  0.858398   
             cellina$_{node-pert=200}$      0.735401  0.197975  0.786953   
             SpatialProp$_{node-pert=200}$  0.700208  0.125093  0.743029   

                                                     direction_match_k  \
                                                 std              mean   
perturbation model_name                                                  
Fiber-tracts mean shift                     0.238833          0.068000   
             CPA                            0.149858          0.309333   
             scGen                          0.201174          0.149333   
             SpatialProp                    0.199540          0.086667   
             MintFlow                       0.180161          0.206667   
             cellina (ablated)              0.141151          0.280000   
             cellina-GAT                    0.157889          0.394286   
             cellina                        0.154841          0.453333   
             cellina$_{node-pert=200}$      0.151456          0.285333   
             SpatialProp$_{node-pert=200}$  0.179516          0.077333   
Isocortex    mean shift                     0.191781          0.250667   
             CPA                            0.172484          0.534667   
             scGen                          0.142527          0.268000   
             SpatialProp                    0.093146          0.088000   
             MintFlow                       0.164817          0.366667   
             cellina (ablated)              0.163441          0.501333   
             cellina-GAT                    0.143880          0.657143   
             cellina                        0.170373          0.588000   
             cellina$_{node-pert=200}$      0.186411          0.350667   
             SpatialProp$_{node-pert=200}$  0.094362          0.082667   

                                                     edistance_pca_log  \
                                                 std              mean   
perturbation model_name                                                  
Fiber-tracts mean shift                     0.069200         24.937697   
             CPA                            0.125554          7.256517   
             scGen                          0.090354          5.340665   
             SpatialProp                    0.074322         20.026668   
             MintFlow                       0.162598         19.951692   


In [10]:
def find_best(agg: pd.DataFrame) -> dict:
    best = {}
    if isinstance(agg.index, pd.MultiIndex):
        # agg index: (perturbation, model_name)
        # collect perturbations preserving order
        raw = [p for p, _ in agg.index.tolist()]
        seen = set()
        perturbations = [p for p in raw if not (p in seen or seen.add(p))]
        for metric, direction in METRICS.items():
            for pert in perturbations:
                try:
                    means = agg.loc[pert][(metric, "mean")].dropna()
                except KeyError:
                    continue
                if means.empty:
                    continue
                best[(pert, metric)] = means.idxmax() if direction > 0 else means.idxmin()
    else:
        for metric, direction in METRICS.items():
            means = agg[(metric, "mean")].dropna()
            if means.empty:
                continue
            best[metric] = means.idxmax() if direction > 0 else means.idxmin()
    return best


def format_cell(mean, std, bold, na_str="--"):
    if pd.isna(mean):
        return na_str
    s = f"{mean:.2f} $\\pm$ {std:.2f}" if not pd.isna(std) else f"{mean:.2f}"
    return f"\\textbf{{{s}}}" if bold else s

In [11]:
best = find_best(agg)
table = pd.DataFrame(index=agg.index, columns=[PRETTY_METRIC[m] for m in METRICS])

for metric in METRICS:
    col = PRETTY_METRIC[metric]
    for model in agg.index:
        pert, model_name = model

        mean = agg.loc[model, (metric, "mean")]
        std = agg.loc[model, (metric, "std")]

        bold = (best.get((pert, metric)) == model_name)

        table.loc[model, col] = format_cell(mean, std, bold)
table

Spearman $\uparrow$  \
perturbation model_name                                                
Fiber-tracts mean shift                              0.35 $\pm$ 0.27   
             CPA                                     0.58 $\pm$ 0.21   
             scGen                                   0.50 $\pm$ 0.23   
             SpatialProp                             0.47 $\pm$ 0.20   
             MintFlow                                0.54 $\pm$ 0.23   
             cellina (ablated)                       0.58 $\pm$ 0.22   
             cellina-GAT                             0.64 $\pm$ 0.15   
             cellina                        \textbf{0.66 $\pm$ 0.21}   
             cellina$_{node-pert=200}$               0.59 $\pm$ 0.23   
             SpatialProp$_{node-pert=200}$           0.50 $\pm$ 0.19   
Isocortex    mean shift                              0.59 $\pm$ 0.19   
             CPA                                     0.78 $\pm$ 0.17   
             scGen                                   0.72 $\pm$ 0.18   
             SpatialProp                             0.70 $\pm$ 0.12   
             MintFlow                                0.77 $\pm$ 0.18   
             cellina (ablated)                       0.79 $\pm$ 0.16   
             cellina-GAT                    \textbf{0.85 $\pm$ 0.11}   
             cellina                                 0.81 $\pm$ 0.18   
             cellina$_{node-pert=200}$               0.74 $\pm$ 0.20   
             SpatialProp$_{node-pert=200}$           0.70 $\pm$ 0.13   

                                                  Pearson $\uparrow$  \
perturbation model_name                                                
Fiber-tracts mean shift                              0.36 $\pm$ 0.24   
             CPA                                     0.80 $\pm$ 0.15   
             scGen                                   0.72 $\pm$ 0.20   
             SpatialProp                             0.64 $\pm$ 0.20   
             MintFlow                                0.78 $\pm$ 0.18   
             cellina (ablated)                       0.80 $\pm$ 0.14   
             cellina-GAT                             0.81 $\pm$ 0.16   
             cellina                        \textbf{0.83 $\pm$ 0.15}   
             cellina$_{node-pert=200}$               0.81 $\pm$ 0.15   
             SpatialProp$_{node-pert=200}$           0.66 $\pm$ 0.18   
Isocortex    mean shift                              0.58 $\pm$ 0.19   
             CPA                                     0.84 $\pm$ 0.17   
             scGen                                   0.82 $\pm$ 0.14   
             SpatialProp                             0.75 $\pm$ 0.09   
             MintFlow                                0.84 $\pm$ 0.16   
             cellina (ablated)                       0.84 $\pm$ 0.16   
             cellina-GAT                    \textbf{0.89 $\pm$ 0.14}   
             cellina                                 0.86 $\pm$ 0.17   
             cellina$_{node-pert=200}$               0.79 $\pm$ 0.19   
             SpatialProp$_{node-pert=200}$           0.74 $\pm$ 0.09   

                                           Signed Precision $\uparrow$  \
perturbation model_name                                                  
Fiber-tracts mean shift                                0.07 $\pm$ 0.07   
             CPA                                       0.31 $\pm$ 0.13   
             scGen                                     0.15 $\pm$ 0.09   
             SpatialProp                               0.09 $\pm$ 0.07   
             MintFlow                                  0.21 $\pm$ 0.16   
             cellina (ablated)                         0.28 $\pm$ 0.14   
             cellina-GAT                               0.39 $\pm$ 0.18   
             cellina                          \textbf{0.45 $\pm$ 0.21}   
             cellina$_{node-pert=200}$                 0.29 $\pm$ 0.19   
             SpatialProp$_{node-pert=200}$             0.08 $\pm$

In [130]:
def insert_midrule_perts(latex, table):
    groups = table.index.get_level_values(0)
    boundaries = [i for i in range(1, len(groups)) if groups[i] != groups[i - 1]]

    lines = latex.splitlines()
    new_lines = []

    row_idx = 0

    for line in lines:
        new_lines.append(line)

        # detect data rows (skip header)
        if "&" in line and "\\\\" in line:
            if row_idx in boundaries:
                new_lines.append(r"\midrule")
            row_idx += 1

    latex = "\n".join(new_lines)
    return latex

In [131]:
is_multi_pert = table.index.get_level_values(0).nunique() > 1

if is_multi_pert:
    latex = table.to_latex(
        index_names=False, # removes the extra header row
        escape=False,
        header=False,
        column_format="ll" + "c" * table.shape[1],  # two index levels → 'll'
        multirow=True,  # for multiple perturbations
        caption=(
            f"Leave-one-celltype-out performance (top {N_DEG} DEGs). For each slide we "
            "first average over the held-out cell types, then report mean $\\pm$ std "
            f"across {df['sid'].nunique()} slides. Best per metric in \\textbf{{bold}}."
        ),
        label=f"tab:loo_summary_{DATASET_NAME}",
    )

    latex = latex.replace(r"\cline{1-7}", "")
    latex = latex.replace("\\begin{table}", "\\begin{table}[t]\n\\centering")
    latex = latex.replace(r"\multirow[t]", r"\multirow[c]")
    header = "Holdout \\\ perturbation & Method & " + " & ".join(table.columns) + r" \\"
    latex = latex.replace(
    "\\begin{tabular}{ll" + "c" * table.shape[1] + "}",
    "\\begin{tabular}{ll" + "c" * table.shape[1] + "}\n\\toprule\n" + header
    )
    latex = latex.replace(r"\midrule", "")
    latex = insert_midrule_perts(latex, table)
else:
    flat = table.reset_index(level=0, drop=True)

    latex = flat.to_latex(
        index=True,
        index_names=False,
        escape=False,
        column_format="l" + "c" * flat.shape[1],
        caption=(
            f"Leave-one-celltype-out performance (top {N_DEG} DEGs). "
            "Mean $\\pm$ std across slides. Best per metric in \\textbf{bold}."
        ),
        label=f"tab:loo_summary_{DATASET_NAME}",
    )
    latex = latex.replace("\\begin{table}", "\\begin{table}[t]\n\\centering")

print(latex)

(OUT_DIR / f"loo_summary_{DATASET_NAME}_DEG_{N_DEG}_table.tex").write_text(latex)

\begin{table}[t]
\centering
\caption{Leave-one-celltype-out performance (top 50 DEGs). For each slide we first average over the held-out cell types, then report mean $\pm$ std across 3 slides. Best per metric in \textbf{bold}.}
\label{tab:loo_summary_merfish}
\begin{tabular}{llcccc}
\toprule
Holdout \\ perturbation & Method & Spearman $\uparrow$ & Pearson $\uparrow$ & Signed Precision $\uparrow$ & RMSE $\downarrow$ \\
\toprule

\multirow[c]{10}{*}{Fiber-tracts} & mean shift & 0.35 $\pm$ 0.27 & 0.36 $\pm$ 0.24 & 0.07 $\pm$ 0.07 & 3.33 $\pm$ 0.30 \\
 & CPA & 0.58 $\pm$ 0.21 & 0.80 $\pm$ 0.15 & 0.31 $\pm$ 0.13 & 3.64 $\pm$ 0.36 \\
 & scGen & 0.50 $\pm$ 0.23 & 0.72 $\pm$ 0.20 & 0.15 $\pm$ 0.09 & 3.53 $\pm$ 0.36 \\
 & SpatialProp & 0.47 $\pm$ 0.20 & 0.64 $\pm$ 0.20 & 0.09 $\pm$ 0.07 & \textbf{3.27 $\pm$ 0.43} \\
 & MintFlow & 0.54 $\pm$ 0.23 & 0.78 $\pm$ 0.18 & 0.21 $\pm$ 0.16 & 3.43 $\pm$ 0.37 \\
 & cellina (ablated) & 0.58 $\pm$ 0.22 & 0.80 $\pm$ 0.14 & 0.28 $\pm$ 0.14 & 3.64 $\pm$ 0.34 \

2469

In [105]:
# Markdown preview (arrows rendered as unicode so they look right in the notebook).
md_metric = {
    "spearman": "Spearman ↑",
    "pearson": "Pearson ↑",
    "precision": "Precision ↑",
    "direction_match_k": "Direction Match (K) ↑",
    "edistance_local": "E-dist (local) ↓",
    "rmse": "RMSE ↓",
    "avg_rank": "Avg. Rank ↓",
}

md_table = pd.DataFrame(index=agg.index, columns=[md_metric[m] for m in METRICS])
for metric in METRICS:
    col = md_metric[metric]
    for model in agg.index:
        mean = agg.loc[model, (metric, "mean")]
        std = agg.loc[model, (metric, "std")]
        if pd.isna(mean):
            md_table.loc[model, col] = "--"
            continue
        s = f"{mean:.3f} ± {std:.3f}"
        md_table.loc[model, col] = f"**{s}**" if best.get(metric) == model else s

md_table.index.name = "Method"
print(md_table.to_markdown())

| Method                                         | Spearman ↑    | Pearson ↑     | Direction Match (K) ↑   | RMSE ↓        |
|:-----------------------------------------------|:--------------|:--------------|:------------------------|:--------------|
| ('Isocortex', 'mean shift')                    | 0.594 ± 0.189 | 0.575 ± 0.192 | 0.251 ± 0.081           | 3.564 ± 0.372 |
| ('Isocortex', 'CPA')                           | 0.782 ± 0.170 | 0.838 ± 0.172 | 0.535 ± 0.137           | 3.594 ± 0.349 |
| ('Isocortex', 'scGen')                         | 0.720 ± 0.183 | 0.820 ± 0.143 | 0.268 ± 0.162           | 3.559 ± 0.371 |
| ('Isocortex', 'MintFlow')                      | 0.772 ± 0.184 | 0.837 ± 0.165 | 0.367 ± 0.161           | 3.774 ± 0.370 |
| ('Isocortex', 'cellina (ablated)')             | 0.789 ± 0.163 | 0.841 ± 0.163 | 0.501 ± 0.139           | 3.626 ± 0.362 |
| ('Isocortex', 'cellina-GAT')                   | 0.853 ± 0.111 | 0.889 ± 0.144 | 0.657 ± 0.099           | 3.559 ± 0.347 |


***